In [19]:
print("Jay Ganesh")

Jay Ganesh


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
import pickle

In [21]:
data=pd.read_csv(r"D:\ProjectAarya\GEN AI\Code\Ch13\Churn_Modelling.csv")

In [22]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
data.drop(['RowNumber','CustomerId','Surname',],axis='columns',inplace=True)

In [24]:
label_gender=LabelEncoder()
data['Gender']=label_gender.fit_transform(data['Gender'])

In [25]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

ohegeo = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  # sparse_output=False avoids sparse issues
geo_encoded = ohegeo.fit_transform(data[['Geography']])  # Full column as DataFrame, no [ ]
geo_encodeddf = pd.DataFrame(geo_encoded, columns=ohegeo.get_feature_names_out())


In [26]:
geo_encodeddf

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [27]:
data=pd.concat([data.drop('Geography',axis='columns'),geo_encodeddf],axis='columns')

In [28]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [29]:
X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']
X

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,1,0.0,1.0,0.0


In [30]:
y

0       101348.88
1       112542.58
2       113931.57
3        93826.63
4        79084.10
          ...    
9995     96270.64
9996    101699.77
9997     42085.58
9998     92888.52
9999     38190.78
Name: EstimatedSalary, Length: 10000, dtype: float64

In [31]:

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.14,random_state=6)
##Scalling the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)


In [32]:
with open('gender.pkl','wb') as f:
    pickle.dump(label_gender,f)
with open('geography.pkl','wb') as f:
    pickle.dump(ohegeo,f)
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

### ANN Regression Problem Statement

In [34]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [35]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1)# Applying no activation function here because we are working with regression task 
])


c:\Users\Jay\anaconda3\envs\ch13\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [36]:
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

In [38]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

log_dir="regressionlogs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [39]:
earlystoppingcallback=EarlyStopping(monitor='val_loss',patience=4,restore_best_weights=True)

In [42]:
history=model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=100,
    callbacks=[earlystoppingcallback,tensorboard_callback]
)

Epoch 1/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 50277.1562 - mae: 50277.1562 - val_loss: 50091.9922 - val_mae: 50091.9922
Epoch 2/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 50211.9727 - mae: 50211.9727 - val_loss: 50048.5586 - val_mae: 50048.5586
Epoch 3/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 50153.6328 - mae: 50153.6328 - val_loss: 50011.4766 - val_mae: 50011.4766
Epoch 4/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 50104.2539 - mae: 50104.2539 - val_loss: 49980.1016 - val_mae: 49980.1016
Epoch 5/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 50059.4609 - mae: 50059.4609 - val_loss: 49960.8906 - val_mae: 49960.8906
Epoch 6/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 50016.7227 - mae: 50016.7227 - val_loss: 49927.7266 - val_mae: 49927.7266
Epoch 7/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 49985.1484 - mae: 49985.1484 - val_loss: 49918.1914 - val_mae: 49918.1914
Epoch 8/100
269/269 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - 

In [43]:
###Evaluate the model

%load_ext tensorboard

In [46]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 5296), started 0:00:46 ago. (Use '!kill 5296' to kill it.)

In [48]:
test_loss,test_mae=model.evaluate(X_test,y_test)

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 49825.1367 - mae: 49825.1367


In [50]:
print(test_loss)

49825.13671875


In [51]:
model.save('regressionmodel.h5')